In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/arockiaselciaa/creditcardcsv/creditcard.csv


In [2]:
import os

os.makedirs('src', exist_ok=True)
os.makedirs('results', exist_ok=True)
print("Directory structure created: src/ and results/")

Directory structure created: src/ and results/


In [3]:
%%writefile src/data_utils.py
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def find_dataset_path(filename="creditcard.csv", search_dir="/kaggle/input"):
    """Dynamically locates the CSV file within Kaggle input directories."""
    for root, dirs, files in os.walk(search_dir):
        if filename in files:
            return os.path.join(root, filename)
    if os.path.exists(filename):
        return filename
    raise FileNotFoundError(f"Could not locate {filename} under {search_dir}. Ensure dataset is added.")

def load_and_preprocess_data(csv_path=None, random_state=42):
    """
    Loads Credit Card Fraud dataset, standardizes Time/Amount features,
    and produces a stratified 80/20 train/test split.
    """
    if csv_path is None:
        csv_path = find_dataset_path()
        
    print(f"Loading data from: {csv_path}")
    df = pd.read_csv(csv_path)

    # Separate features and target
    X = df.drop(columns=['Class'])
    y = df['Class'].values

    # Stratified Train/Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=random_state
    )

    # Scale 'Time' and 'Amount' fitting strictly on the training set
    scaler = StandardScaler()
    scale_cols = ['Time', 'Amount']

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
    X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

    return X_train_scaled.values, X_test_scaled.values, y_train, y_test

def get_svm_subsample(X_train, y_train, n_negatives=3000, random_state=42):
    """
    Subsamples a manageable dataset specifically for SMO-SVM training.
    Retains all fraud instances and samples a fixed subset of non-fraud instances.
    """
    np.random.seed(random_state)

    pos_mask = (y_train == 1)
    neg_mask = (y_train == 0)

    X_pos, y_pos = X_train[pos_mask], y_train[pos_mask]
    X_neg, y_neg = X_train[neg_mask], y_train[neg_mask]

    neg_indices = np.random.choice(len(X_neg), size=min(n_negatives, len(X_neg)), replace=False)
    X_neg_sub, y_neg_sub = X_neg[neg_indices], y_neg[neg_indices]

    X_sub = np.vstack([X_pos, X_neg_sub])
    y_sub = np.hstack([y_pos, y_neg_sub])

    shuffle_idx = np.random.permutation(len(y_sub))
    return X_sub[shuffle_idx], y_sub[shuffle_idx]

Writing src/data_utils.py


In [4]:
import sys
sys.path.append('src')
from data_utils import load_and_preprocess_data, get_svm_subsample

# Load Full Dataset
X_train, X_test, y_train, y_test = load_and_preprocess_data()
print(f"Full Train Shape: {X_train.shape} | Positives: {y_train.sum()} ({y_train.sum()/len(y_train):.4%})")
print(f"Full Test Shape:  {X_test.shape}  | Positives: {y_test.sum()} ({y_test.sum()/len(y_test):.4%})")

# Generate SVM Subsample
X_svm_train, y_svm_train = get_svm_subsample(X_train, y_train, n_negatives=3000)
print(f"\nSVM Subsample Train Shape: {X_svm_train.shape} | Positives: {y_svm_train.sum()}")

Loading data from: /kaggle/input/datasets/arockiaselciaa/creditcardcsv/creditcard.csv
Full Train Shape: (227845, 30) | Positives: 394 (0.1729%)
Full Test Shape:  (56962, 30)  | Positives: 98 (0.1720%)

SVM Subsample Train Shape: (3394, 30) | Positives: 394


In [5]:
%%writefile src/baselines.py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

FEATURE_NAMES = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']

def run_logistic_regression_sweep(X_train, y_train, C_values=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0]):
    """
    Sweeps C values for L1 (liblinear) and L2 (lbfgs) Logistic Regression.
    Returns fitted models and weight histories for path plotting.
    """
    l1_weights = []
    l2_weights = []
    
    l1_models = {}
    l2_models = {}

    for C in C_values:
        # L1 Penalty (Sparse, Laplace Prior)
        clf_l1 = LogisticRegression(penalty='l1', C=C, solver='liblinear', random_state=42, max_iter=1000)
        clf_l1.fit(X_train, y_train)
        l1_weights.append(clf_l1.coef_[0].copy())
        l1_models[C] = clf_l1

        # L2 Penalty (Shrinkage, Gaussian Prior)
        clf_l2 = LogisticRegression(penalty='l2', C=C, solver='lbfgs', random_state=42, max_iter=1000)
        clf_l2.fit(X_train, y_train)
        l2_weights.append(clf_l2.coef_[0].copy())
        l2_models[C] = clf_l2

    return C_values, np.array(l1_weights), np.array(l2_weights), l1_models, l2_models

def plot_regularization_path(C_values, l1_weights, l2_weights, save_path="results/regularization_path.png"):
    """Plots L1 vs L2 coefficient paths as a function of log10(C)."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    log_C = np.log10(C_values)

    # L1 Path
    axes[0].plot(log_C, l1_weights, alpha=0.7)
    axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
    axes[0].set_title("L1 Regularization Path (Laplace Prior - Sparse)")
    axes[0].set_xlabel(r"$\log_{10}(C)$")
    axes[0].set_ylabel("Coefficient Value")
    axes[0].grid(True, alpha=0.3)

    # L2 Path
    axes[1].plot(log_C, l2_weights, alpha=0.7)
    axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
    axes[1].set_title("L2 Regularization Path (Gaussian Prior - Smooth Shrinkage)")
    axes[1].set_xlabel(r"$\log_{10}(C)$")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Regularization path saved to {save_path}")

def report_l1_feature_selection(C_values, l1_weights):
    """Reports feature elimination order under strong L1 regularization."""
    print("\n--- L1 Feature Selection Analysis ---")
    for idx, C in enumerate(C_values):
        weights = l1_weights[idx]
        zero_cols = [FEATURE_NAMES[i] for i in range(len(weights)) if weights[i] == 0.0]
        non_zero_count = len(weights) - len(zero_cols)
        print(f"C = {C:<6} | Active Features: {non_zero_count}/30 | Zeroed out ({len(zero_cols)}): {zero_cols[:5]}...")

def train_naive_bayes(X_train, y_train):
    """Fits Gaussian Naive Bayes baseline."""
    gnb = GaussianNB()
    gnb.fit(X_train, y_train)
    return gnb

Writing src/baselines.py


In [6]:
import sys
import importlib

if 'src' not in sys.path:
    sys.path.append('src')

import baselines
importlib.reload(baselines)
from baselines import run_logistic_regression_sweep, plot_regularization_path, report_l1_feature_selection, train_naive_bayes

# 1. Run Logistic Regression Sweep
C_values, l1_weights, l2_weights, l1_models, l2_models = run_logistic_regression_sweep(X_train, y_train)

# 2. Plot Regularization Path
plot_regularization_path(C_values, l1_weights, l2_weights)

# 3. Report L1 Sparsity/Feature Elimination
report_l1_feature_selection(C_values, l1_weights)

# 4. Train Gaussian Naive Bayes
gnb_model = train_naive_bayes(X_train, y_train)
print("\nGaussian Naive Bayes baseline successfully trained.")

Regularization path saved to results/regularization_path.png

--- L1 Feature Selection Analysis ---
C = 0.001  | Active Features: 6/30 | Zeroed out (24): ['Time', 'V1', 'V2', 'V4', 'V5']...
C = 0.01   | Active Features: 9/30 | Zeroed out (21): ['Time', 'V1', 'V2', 'V3', 'V6']...
C = 0.1    | Active Features: 19/30 | Zeroed out (11): ['Time', 'V1', 'V7', 'V11', 'V12']...
C = 1.0    | Active Features: 28/30 | Zeroed out (2): ['V19', 'V26']...
C = 10.0   | Active Features: 30/30 | Zeroed out (0): []...
C = 100.0  | Active Features: 30/30 | Zeroed out (0): []...

Gaussian Naive Bayes baseline successfully trained.


In [7]:
%%writefile src/smo_svm.py
import numpy as np

class SMOSVM:
    """
    Support Vector Machine trained via Sequential Minimal Optimization (SMO).
    Internally handles {0, 1} to {-1, +1} label math conversion.
    """
    def __init__(self, C=1.0, pos_weight=1.0, kernel='linear', gamma=None, tol=1e-3, max_passes=5, max_iter=500):
        self.C = C
        self.pos_weight = pos_weight
        self.kernel_type = kernel
        self.gamma = gamma
        self.tol = tol
        self.max_passes = max_passes
        self.max_iter = max_iter
        
        self.alpha = None
        self.b = 0.0
        self.support_vectors = None
        self.support_vector_labels = None
        self.support_vector_alphas = None

    def _kernel(self, X1, X2):
        if self.kernel_type == 'linear':
            return np.dot(X1, X2.T)
        elif self.kernel_type == 'rbf':
            if self.gamma is None:
                self.gamma = 1.0 / X1.shape[1]
            if X1.ndim == 1:
                X1 = X1.reshape(1, -1)
            if X2.ndim == 1:
                X2 = X2.reshape(1, -1)
            dist_sq = np.sum(X1**2, axis=1, keepdims=True) + np.sum(X2**2, axis=1) - 2 * np.dot(X1, X2.T)
            return np.exp(-self.gamma * dist_sq)
        else:
            raise ValueError(f"Unsupported kernel: {self.kernel_type}")

    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # --- LABEL FIX: Map {0, 1} to {-1, +1} purely for internal math ---
        y_math = np.where(y == 0, -1, 1).astype(np.float64)
        
        K = self._kernel(X, X)
        self.alpha = np.zeros(n_samples)
        self.b = 0.0
        
        C_bounds = np.where(y_math == 1, self.C * self.pos_weight, self.C)
        
        passes = 0
        iters = 0

        while passes < self.max_passes and iters < self.max_iter:
            num_changed_alphas = 0
            for i in range(n_samples):
                f_i = np.dot(self.alpha * y_math, K[:, i]) + self.b
                E_i = f_i - y_math[i]
                
                r_i = E_i * y_math[i]
                if (r_i < -self.tol and self.alpha[i] < C_bounds[i]) or (r_i > self.tol and self.alpha[i] > 0):
                    j = np.random.choice([idx for idx in range(n_samples) if idx != i])
                    
                    f_j = np.dot(self.alpha * y_math, K[:, j]) + self.b
                    E_j = f_j - y_math[j]
                    
                    alpha_i_old = self.alpha[i]
                    alpha_j_old = self.alpha[j]
                    
                    if y_math[i] != y_math[j]:
                        L = max(0.0, self.alpha[j] - self.alpha[i])
                        H = min(C_bounds[j], C_bounds[i] + self.alpha[j] - self.alpha[i])
                    else:
                        L = max(0.0, self.alpha[i] + self.alpha[j] - C_bounds[i])
                        H = min(C_bounds[j], self.alpha[i] + self.alpha[j])
                        
                    if L == H: continue
                        
                    eta = 2.0 * K[i, j] - K[i, i] - K[j, j]
                    if eta >= 0: continue
                        
                    self.alpha[j] -= (y_math[j] * (E_i - E_j)) / eta
                    self.alpha[j] = min(H, max(L, self.alpha[j]))
                    
                    if abs(self.alpha[j] - alpha_j_old) < 1e-5: continue
                        
                    self.alpha[i] += y_math[i] * y_math[j] * (alpha_j_old - self.alpha[j])
                    
                    b1 = self.b - E_i - y_math[i] * (self.alpha[i] - alpha_i_old) * K[i, i] - y_math[j] * (self.alpha[j] - alpha_j_old) * K[i, j]
                    b2 = self.b - E_j - y_math[i] * (self.alpha[i] - alpha_i_old) * K[i, j] - y_math[j] * (self.alpha[j] - alpha_j_old) * K[j, j]
                    
                    if 0 < self.alpha[i] < C_bounds[i]:
                        self.b = b1
                    elif 0 < self.alpha[j] < C_bounds[j]:
                        self.b = b2
                    else:
                        self.b = (b1 + b2) / 2.0
                        
                    num_changed_alphas += 1

            iters += 1
            if num_changed_alphas == 0:
                passes += 1
            else:
                passes = 0

        sv_mask = self.alpha > 1e-5
        self.support_vectors = X[sv_mask]
        self.support_vector_labels = y_math[sv_mask]
        self.support_vector_alphas = self.alpha[sv_mask]
        
        print(f"SMO Converged in {iters} iterations. Support Vectors: {len(self.support_vectors)} / {n_samples}")

    def decision_function(self, X):
        if self.support_vectors is None or len(self.support_vectors) == 0:
            return np.zeros(X.shape[0])
        K_sv = self._kernel(X, self.support_vectors)
        return np.dot(K_sv, self.support_vector_alphas * self.support_vector_labels) + self.b

    def predict(self, X):
        # NATIVE OUTPUT MAP: Margin >= 0 is Class 1, else Class 0
        scores = self.decision_function(X)
        return np.where(scores >= 0, 1, 0)

Writing src/smo_svm.py


In [8]:
import sys
import importlib
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

if 'src' not in sys.path:
    sys.path.append('src')

import smo_svm
importlib.reload(smo_svm)
from smo_svm import SMOSVM


# 1. Fit Custom SMO-SVM Engine
print("--- Fitting Custom SMO-SVM Engine ---")
custom_svm = SMOSVM(C=1.0, kernel='linear', max_passes=5, max_iter=500)
custom_svm.fit(X_svm_train, y_svm_train)  # <-- Passes standard 0/1 labels

# 2. Fit Reference Scikit-Learn SVC
print("\n--- Fitting Reference sklearn.svm.SVC ---")
sk_svm = SVC(C=1.0, kernel='linear')
sk_svm.fit(X_svm_train, y_svm_train)  # <-- Passes standard 0/1 labels

# 3. Predictions (Both now naturally output {0, 1})
custom_preds = custom_svm.predict(X_test)
sk_preds = sk_svm.predict(X_test)

# 4. Verification Analysis
print("\n=== VERIFICATION BENCHMARK REPORT ===")
print(f"Custom SMO Accuracy:   {accuracy_score(y_test, custom_preds):.5f}")
print(f"Scikit-Learn Accuracy: {accuracy_score(y_test, sk_preds):.5f}")
print(f"Prediction Agreement:  {np.mean(custom_preds == sk_preds):.4%}")
print(f"Custom Support Vectors:  {len(custom_svm.support_vectors)}")
print(f"Sklearn Support Vectors: {sk_svm.n_support_.sum()}")

# 5. Weight Verification Check
w_custom = np.dot(custom_svm.support_vector_alphas * custom_svm.support_vector_labels, custom_svm.support_vectors)
w_sklearn = sk_svm.coef_[0]
cosine_sim = np.dot(w_custom, w_sklearn) / (np.linalg.norm(w_custom) * np.linalg.norm(w_sklearn))
print(f"Weight Vector Cosine Similarity: {cosine_sim:.5f}")

--- Fitting Custom SMO-SVM Engine ---
SMO Converged in 500 iterations. Support Vectors: 359 / 3394

--- Fitting Reference sklearn.svm.SVC ---

=== VERIFICATION BENCHMARK REPORT ===
Custom SMO Accuracy:   0.99598
Scikit-Learn Accuracy: 0.99858
Prediction Agreement:  99.7367%
Custom Support Vectors:  359
Sklearn Support Vectors: 162
Weight Vector Cosine Similarity: 0.93393


In [9]:
%%writefile src/evaluation.py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    precision_recall_curve, auc, roc_curve
)

def evaluate_model(y_true, y_pred, y_scores, model_name="Model"):
    """Calculates evaluation metrics for binary classification."""
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    roc_auc = roc_auc_score(y_true, y_scores) if y_scores is not None else np.nan
    
    if y_scores is not None:
        p_curve, r_curve, _ = precision_recall_curve(y_true, y_scores)
        pr_auc = auc(r_curve, p_curve)
    else:
        pr_auc = np.nan

    return {
        "Model": model_name,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    }

def plot_performance_curves(results_dict, y_true, save_path="results/performance_curves.png"):
    """Plots PR and ROC curves for all models side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for model_name, scores in results_dict.items():
        if scores is None:
            continue
            
        # PR Curve
        p, r, _ = precision_recall_curve(y_true, scores)
        pr_auc = auc(r, p)
        axes[0].plot(r, p, label=f"{model_name} (PR-AUC = {pr_auc:.4f})")

        # ROC Curve
        fpr, tpr, _ = roc_curve(y_true, scores)
        roc_auc = roc_auc_score(y_true, scores)
        axes[1].plot(fpr, tpr, label=f"{model_name} (ROC-AUC = {roc_auc:.4f})")

    # Baseline PR threshold line (ratio of positives)
    baseline_pr = y_true.sum() / len(y_true)
    axes[0].axhline(baseline_pr, color='red', linestyle='--', label=f'Random Baseline ({baseline_pr:.4f})')
    axes[0].set_title("Precision-Recall Curve (Primary Imbalance Metric)")
    axes[0].set_xlabel("Recall")
    axes[0].set_ylabel("Precision")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc="lower left")

    # Diagonal ROC line
    axes[1].plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Chance (0.5000)')
    axes[1].set_title("ROC Curve")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc="lower right")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Performance curves saved to {save_path}")

Writing src/evaluation.py


In [10]:
import sys
import importlib
import pandas as pd
import numpy as np

if 'src' not in sys.path:
    sys.path.append('src')

import evaluation
importlib.reload(evaluation)
from evaluation import evaluate_model, plot_performance_curves

# Retrieve best tuned models from Phase 2 & 3
best_l1_logreg = l1_models[0.1]
best_l2_logreg = l2_models[0.1]

# Gather Continuous Decision Scores for Curves
scores_dict = {
    "LogReg (L1 Penalty)": best_l1_logreg.decision_function(X_test),
    "LogReg (L2 Penalty)": best_l2_logreg.decision_function(X_test),
    "Gaussian Naive Bayes": gnb_model.predict_proba(X_test)[:, 1],
    "Custom SMO-SVM (Linear)": custom_svm.decision_function(X_test)
}

# Generate Predictions & Metrics
metrics_list = []

# 1. L1 Logistic Regression
preds_l1 = best_l1_logreg.predict(X_test)
metrics_list.append(evaluate_model(y_test, preds_l1, scores_dict["LogReg (L1 Penalty)"], "LogReg (L1)"))

# 2. L2 Logistic Regression
preds_l2 = best_l2_logreg.predict(X_test)
metrics_list.append(evaluate_model(y_test, preds_l2, scores_dict["LogReg (L2 Penalty)"], "LogReg (L2)"))

# 3. Gaussian Naive Bayes
preds_gnb = gnb_model.predict(X_test)
metrics_list.append(evaluate_model(y_test, preds_gnb, scores_dict["Gaussian Naive Bayes"], "Gaussian NB"))

# 4. Custom SMO-SVM
preds_smo = custom_svm.predict(X_test)
metrics_list.append(evaluate_model(y_test, preds_smo, scores_dict["Custom SMO-SVM (Linear)"], "Custom SMO-SVM"))

# Display Benchmark Table
df_results = pd.DataFrame(metrics_list)
print("\n=== PHASE 4 MODEL BENCHMARK TABLE ===")
print(df_results.to_string(index=False))

# Plot and Save PR and ROC Curves
plot_performance_curves(scores_dict, y_test)


=== PHASE 4 MODEL BENCHMARK TABLE ===
         Model  Precision   Recall  F1-Score  ROC-AUC   PR-AUC
   LogReg (L1)   0.826667 0.632653  0.716763 0.965377 0.741223
   LogReg (L2)   0.828947 0.642857  0.724138 0.959524 0.740129
   Gaussian NB   0.058782 0.846939  0.109934 0.963183 0.424035
Custom SMO-SVM   0.282392 0.867347  0.426065 0.968825 0.730301
Performance curves saved to results/performance_curves.png


In [11]:
import os
os.makedirs('src', exist_ok=True)

In [12]:
%%writefile src/phase5_sweep.py
import numpy as np
import pandas as pd
import time
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, auc

def run_weight_sweep(svm_class, X_train, y_train, X_test, y_test, weights=[0.1, 0.5, 1.0, 5.0, 10.0]):
    print("=== PHASE 5: ASYMMETRIC WEIGHT SWEEP ===")
    results = []
    
    for w in weights:
        print(f"Training Custom SMO-SVM with pos_weight={w}...")
        start_time = time.time()
        
        # Initialize custom SVM with the specific positive weight
        svm = svm_class(C=1.0, pos_weight=w, kernel='linear', max_passes=3, max_iter=400)
        svm.fit(X_train, y_train)
        
        # Predictions
        preds = svm.predict(X_test)
        scores = svm.decision_function(X_test)
        
        # Metrics
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        f1 = f1_score(y_test, preds, zero_division=0)
        
        p_curve, r_curve, _ = precision_recall_curve(y_test, scores)
        pr_auc = auc(r_curve, p_curve)
        
        elapsed = time.time() - start_time
        
        results.append({
            "Pos Weight (C+)": w,
            "Precision": round(prec, 5),
            "Recall": round(rec, 5),
            "F1-Score": round(f1, 5),
            "PR-AUC": round(pr_auc, 5),
            "Time (s)": round(elapsed, 2)
        })
        
    return pd.DataFrame(results)

Writing src/phase5_sweep.py


In [13]:
import sys
import importlib
import pandas as pd

if 'src' not in sys.path:
    sys.path.append('src')

import smo_svm
import phase5_sweep
importlib.reload(smo_svm)
importlib.reload(phase5_sweep)

from smo_svm import SMOSVM
from phase5_sweep import run_weight_sweep

# Run the sweep (we test weights below 1.0 to tighten the boundary and increase precision)
sweep_weights = [0.1, 0.3, 0.5, 1.0, 2.0]
df_sweep = run_weight_sweep(SMOSVM, X_svm_train, y_svm_train, X_test, y_test, weights=sweep_weights)

print("\n=== FINAL SWEEP RESULTS ===")
print(df_sweep.to_string(index=False))

# Identify the best weight based on F1-Score
best_row = df_sweep.loc[df_sweep['F1-Score'].idxmax()]
print(f"\nBest configuration found at pos_weight={best_row['Pos Weight (C+)']} with F1-Score: {best_row['F1-Score']}")

=== PHASE 5: ASYMMETRIC WEIGHT SWEEP ===
Training Custom SMO-SVM with pos_weight=0.1...
SMO Converged in 400 iterations. Support Vectors: 296 / 3394
Training Custom SMO-SVM with pos_weight=0.3...
SMO Converged in 400 iterations. Support Vectors: 292 / 3394
Training Custom SMO-SVM with pos_weight=0.5...
SMO Converged in 400 iterations. Support Vectors: 368 / 3394
Training Custom SMO-SVM with pos_weight=1.0...
SMO Converged in 400 iterations. Support Vectors: 417 / 3394
Training Custom SMO-SVM with pos_weight=2.0...
SMO Converged in 400 iterations. Support Vectors: 448 / 3394

=== FINAL SWEEP RESULTS ===
 Pos Weight (C+)  Precision  Recall  F1-Score  PR-AUC  Time (s)
             0.1    0.64062 0.83673   0.72566 0.73452    121.44
             0.3    0.43367 0.86735   0.57823 0.73630    137.46
             0.5    0.41709 0.84694   0.55892 0.72744    147.76
             1.0    0.31159 0.87755   0.45989 0.72716    163.31
             2.0    0.17576 0.88776   0.29342 0.74491    178.58

Best 